In [ ]:
# ============================================================================
# GEO FOUNDATION MODEL — AMOSTRAGEM E EXTRAÇÃO DE DATASET
# ============================================================================
# Objetivo: construir dataset de treinamento para o GeoFM
#   baseado em séries temporais MapBiomas col 10.1 (1985-2024)
#   com 6 classes agregadas e estratificação por células 2x2 graus
#
# Classes agregadas (5 + T):
#   N: vegetação nativa   {1,3,4,5,6,49,10,11,12,32,29,50}
#      Floresta: 1,3,4,5,6,49
#      Herb/Arbustiva: 10,11,12,32,29,50
#   P: pastagem           {15}
#   A: agricultura+silv.  {9,14,18,19,20,35,36,39,40,41,46,47,48,62}
#      Silvicultura: 9 | Pais: 14,18
#      Temp: 19,39,20,40,62,41 | Perene: 36,46,47,35,48
#   U: não vegetado       {22,23,24,25,30,75}
#      inclui Fotovoltaica (75) e Mineração (30)
#   W: água               {26,33,31}
#      Rio/Lago: 33 | Aquicultura: 31 | Pai: 26
#   T: transição/mosaico  {21} — mesma abordagem P→S
#
#   NODATA: classe 27 (Não Observado) → 0 (não mapear)
#
# Erros corrigidos vs versão anterior:
#   27 (Not Observed) removido de W → NODATA
#   28 (inexistente na col10) removido de W
#   29 (Rocky Outcrop) movido de W → N (vegetação herb/arbustiva)
#   31 (Aquicultura) movido de U → W
#   33 (Rio/Lago) movido de U → W
#   35 (Palm Oil) adicionado em A
#   62 (Cotton beta) adicionado em A
#   75 (Fotovoltaica beta) adicionado em U
#
# Processos de interesse:
#   1. N → outra classe      (saída de nativa — desmatamento)
#   2. P → A                 (intensificação agrícola)
#   3. Estabilidade          (permanência em qualquer classe)
#   4. N ↔ outras            (alternância — conversão incompleta)
#   5. Outras → N            (regeneração — censurado à direita)
#
# Estratificação:
#   80 células de 2x2 graus cobrindo o Brasil
#   Estratificadas por bioma dominante
#   Filtro: cobertura >= 50% da área do Brasil
# ============================================================================

import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window, from_bounds
from rasterio.transform import from_bounds as transform_from_bounds
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
import json
from datetime import datetime

# ── Caminhos ─────────────────────────────────────────────────────────────────
GRID_XLSX   = r"C:\Users\mario.barroso\Documents\ArcGIS\Projects\teste3\grid_2x2_wgs84_TableToExcel.xlsx"
RASTER_DIR  = r"E:\Dados\MapBiomas\LULC_col101"
RASTER_PAT  = "{ano}_coverage_lclu_1-1-1_.tif"
OUT_DIR     = Path(r"D:\Projetos\GeoFM")
OUT_DIR.mkdir(exist_ok=True, parents=True)

YEARS       = list(range(1985, 2025))   # 40 anos
SEED        = 2026
N_CELLS     = 80
MIN_COV     = 50   # cobertura mínima %
NODATA      = 0

# ── Mapeamento de classes → 5 classes + T ────────────────────────────────────
# Baseado na legenda oficial MapBiomas Coleção 10 (PDF)
# Classe 21 (mosaico) mantida como T (transição) — mesma abordagem do P→S
# Classe 27 (Não Observado) → NODATA (0) — não incluída no mapa
CLASS_MAP = {
    # N — Vegetação Nativa
    # Floresta: pai(1), Florestal(3), Savânica(4), Mangue(5),
    #           Alagável(6), Restinga Arbórea(49)
    1:'N', 3:'N', 4:'N', 5:'N', 6:'N', 49:'N',
    # Herb/Arbustiva: pai(10), Campo Alagado(11), Campestre(12),
    #                 Apicum(32), Afloramento Rochoso(29), Restinga Herb(50)
    10:'N', 11:'N', 12:'N', 32:'N', 29:'N', 50:'N',
    # P — Pastagem
    15:'P',
    # A — Agricultura + Silvicultura
    # Silvicultura(9), pais(14,18)
    9:'A', 14:'A', 18:'A',
    # Temp: Lavoura(19), Soja(39), Cana(20), Arroz(40),
    #       Algodão beta(62), Outras(41)
    19:'A', 39:'A', 20:'A', 40:'A', 62:'A', 41:'A',
    # Perene: pai(36), Café(46), Citrus(47), Dendê(35), Outras(48)
    36:'A', 46:'A', 47:'A', 35:'A', 48:'A',
    # U — Área Não Vegetada
    # pai(22), Praia/Duna(23), Urbano(24), Outras(25),
    # Mineração(30), Fotovoltaica beta(75)
    22:'U', 23:'U', 24:'U', 25:'U', 30:'U', 75:'U',
    # W — Corpo D'água
    # pai(26), Rio/Lago/Oceano(33), Aquicultura(31)
    26:'W', 33:'W', 31:'W',
    # T — Transição/Mosaico (classe 21 — não forma classe M)
    21:'T',
    # 27 (Não Observado) → NÃO MAPEADO → permanece 0 (NODATA)
}
CLASS_CODES = {'N':1, 'P':2, 'A':3, 'U':4, 'W':5, 'T':6}
CLASS_NAMES = {v:k for k,v in CLASS_CODES.items()}

# Vetor de mapeamento rápido
MAP_VECTOR = np.zeros(256, dtype=np.uint8)
for orig, name in CLASS_MAP.items():
    MAP_VECTOR[orig] = CLASS_CODES[name]

print("✅ Configuração GeoFM")
print(f"   Rasters: {RASTER_DIR}")
print(f"   Período: {YEARS[0]}-{YEARS[-1]} ({len(YEARS)} anos)")
print(f"   Células: {N_CELLS} (cobertura >= {MIN_COV}%)")
print(f"   Output:  {OUT_DIR}")
print()
print("Classes agregadas:")
for name, code in CLASS_CODES.items():
    orig = [k for k,v in CLASS_MAP.items() if v==name]
    print(f"  {name} ({code}): classes MapBiomas {sorted(orig)}")

In [ ]:
# ============================================================================
# AMOSTRAGEM ESTRATIFICADA POR BIOMA
# ============================================================================

# Carregar grade
df_grid = pd.read_excel(GRID_XLSX)
print(f"Grade total: {len(df_grid)} células")

# Filtrar por cobertura
df_elig = df_grid[df_grid['Cobertura'] >= MIN_COV].copy()
print(f"Elegíveis (cob >= {MIN_COV}%): {len(df_elig)} células")

# Bioma dominante (primeiro elemento antes do '-')
df_elig['bioma_dom'] = df_elig['bioma'].apply(lambda b: b.split('-')[0])

print(f"\nDistribuição por bioma dominante:")
print(df_elig['bioma_dom'].value_counts().to_string())

# Estratégia de amostragem:
# Proporcional ao número de células elegíveis por bioma
# Mínimo de 2 células por bioma representado
# Total = N_CELLS
biome_counts = df_elig['bioma_dom'].value_counts()
total_elig   = len(df_elig)

# Calcular n por bioma
n_per_biome = {}
for bioma, count in biome_counts.items():
    n = max(2, round(count/total_elig * N_CELLS))
    n_per_biome[bioma] = min(n, count)  # não pode exceder elegíveis

# Ajustar para totalizar N_CELLS
total_proposed = sum(n_per_biome.values())
diff = N_CELLS - total_proposed
if diff != 0:
    # Ajustar no maior bioma
    maior = max(n_per_biome, key=n_per_biome.get)
    n_per_biome[maior] += diff
    n_per_biome[maior] = min(n_per_biome[maior], biome_counts[maior])

print(f"\nAmostragem proposta ({sum(n_per_biome.values())} células):")
print(f"{'Bioma':<8} {'Elegíveis':>10} {'Sortear':>10} {'Proporção':>10}")
print("-" * 42)
for bioma, n in sorted(n_per_biome.items()):
    elig = biome_counts[bioma]
    print(f"{bioma:<8} {elig:>10} {n:>10} {100*n/N_CELLS:>9.1f}%")
print(f"{'TOTAL':<8} {total_elig:>10} {sum(n_per_biome.values()):>10}")

# Sortear células por bioma
np.random.seed(SEED)
sampled_cells = []
for bioma, n in n_per_biome.items():
    subset = df_elig[df_elig['bioma_dom'] == bioma]
    selected = subset.sample(n=n, random_state=SEED)
    sampled_cells.append(selected)

df_sample = pd.concat(sampled_cells, ignore_index=True)
df_sample = df_sample.sort_values(['ROW','COL']).reset_index(drop=True)

print(f"\n✅ {len(df_sample)} células sorteadas")
print(f"\nCélulas selecionadas:")
print(df_sample[['CELL_ID','bioma','Cobertura',
                  'LAT_MIN','LAT_MAX','LON_MIN','LON_MAX']].to_string())

# Salvar
df_sample.to_csv(OUT_DIR / f"sampled_cells_n{N_CELLS}_seed{SEED}.csv",
                 index=False)
print(f"\n✅ sampled_cells_n{N_CELLS}_seed{SEED}.csv")

In [ ]:
# ============================================================================
# VERIFICAÇÃO — rasters disponíveis e sobreposição com células
# ============================================================================

import rasterio

# Verificar primeiro e último ano
for ano in [YEARS[0], YEARS[-1]]:
    path = os.path.join(RASTER_DIR, RASTER_PAT.format(ano=ano))
    if os.path.exists(path):
        with rasterio.open(path) as ds:
            print(f"✅ {ano}: {ds.width}x{ds.height} pixels, CRS={ds.crs.to_epsg()}")
            print(f"   Bounds: {ds.bounds}")
            raster_bounds = ds.bounds
    else:
        print(f"❌ {ano}: arquivo não encontrado em {path}")

print()

# Verificar sobreposição de todas as células com o raster
n_ok = 0
n_fail = 0
for _, row in df_sample.iterrows():
    cell_lon_min = row['LON_MIN']
    cell_lon_max = row['LON_MAX']
    cell_lat_min = row['LAT_MIN']
    cell_lat_max = row['LAT_MAX']

    # Verificar sobreposição com bounds do raster
    overlap = (
        cell_lon_min < raster_bounds.right and
        cell_lon_max > raster_bounds.left and
        cell_lat_min < raster_bounds.top and
        cell_lat_max > raster_bounds.bottom
    )
    if overlap:
        n_ok += 1
    else:
        n_fail += 1
        print(f"⚠️  Sem sobreposição: {row['CELL_ID']}")

print(f"Células com sobreposição válida: {n_ok}/{len(df_sample)}")
if n_fail == 0:
    print("✅ Todas as células cobrem área do raster")

In [ ]:
# ============================================================================
# EXTRAÇÃO DE SÉRIES TEMPORAIS POR CÉLULA
# ============================================================================
# Para cada célula sorteada:
#   1. Extrair janela do raster (todos os anos)
#   2. Reclassificar para 6 classes
#   3. Calcular estatísticas de transição por pixel
#   4. Salvar série temporal comprimida
#
# Output por célula: array (n_pixels, n_anos) com classes agregadas
# Formato: numpy compressed (.npz) — leve e rápido de carregar

SERIES_DIR = OUT_DIR / "series_temporais"
SERIES_DIR.mkdir(exist_ok=True)

def extrair_serie_celula(cell_id, lon_min, lon_max, lat_min, lat_max):
    """
    Extrai série temporal de uma célula 2x2 graus.
    Retorna array (n_pixels_validos, n_anos+2) onde:
      col 0: row no raster original
      col 1: col no raster original
      col 2+: classe agregada por ano (1985-2024)
    """
    series_list = []
    rows_arr = None
    cols_arr = None
    mask_valid = None

    for i, ano in enumerate(YEARS):
        path = os.path.join(RASTER_DIR, RASTER_PAT.format(ano=ano))
        with rasterio.open(path) as ds:
            # Converter bounds geográficos para window do raster
            window = from_bounds(
                left=lon_min, bottom=lat_min,
                right=lon_max, top=lat_max,
                transform=ds.transform
            )
            # Garantir limites válidos
            window = window.intersection(
                Window(0, 0, ds.width, ds.height))
            if window.width <= 0 or window.height <= 0:
                return None

            data = ds.read(1, window=window, out_dtype='uint8')

            if i == 0:
                H, W = data.shape
                row_off = int(window.row_off)
                col_off = int(window.col_off)
                rows_g, cols_g = np.mgrid[
                    row_off:row_off+H, col_off:col_off+W]
                rows_arr = rows_g.flatten()
                cols_arr = cols_g.flatten()
                # Máscara: pixels com pelo menos 1 classe válida
                mask_valid = np.ones(H*W, dtype=bool)

            # Reclassificar
            reclassed = MAP_VECTOR[data].flatten()
            series_list.append(reclassed)

    if not series_list:
        return None

    series = np.stack(series_list, axis=1)  # (n_pixels, n_anos)

    # Filtrar pixels com pelo menos 10 anos de dados válidos
    n_valid = (series > 0).sum(axis=1)
    mask = n_valid >= 10
    if mask.sum() == 0:
        return None

    return {
        'row':    rows_arr[mask],
        'col':    cols_arr[mask],
        'series': series[mask].astype(np.uint8),
        'anos':   np.array(YEARS, dtype=np.int16)
    }


print(f"Extraindo séries temporais para {len(df_sample)} células...")
print(f"⚠️  Pode demorar 30-90 min dependendo do tamanho das células")
print()

stats = []
for _, row in tqdm(df_sample.iterrows(), total=len(df_sample),
                   desc="Células"):
    cell_id  = row['CELL_ID']
    out_path = SERIES_DIR / f"{cell_id}.npz"

    # Pular se já extraído
    if out_path.exists():
        data = np.load(out_path)
        n_px = len(data['row'])
        stats.append({'cell_id':cell_id,'n_pixels':n_px,'status':'cached'})
        continue

    result = extrair_serie_celula(
        cell_id,
        row['LON_MIN'], row['LON_MAX'],
        row['LAT_MIN'], row['LAT_MAX']
    )

    if result is None:
        stats.append({'cell_id':cell_id,'n_pixels':0,'status':'failed'})
        print(f"  ⚠️  {cell_id}: sem dados válidos")
        continue

    np.savez_compressed(out_path, **result)
    n_px = len(result['row'])
    stats.append({'cell_id':cell_id,'n_pixels':n_px,'status':'ok'})

df_stats = pd.DataFrame(stats)
df_stats.to_csv(OUT_DIR / "extraction_stats.csv", index=False)

print(f"\n✅ Extração concluída!")
print(f"   Células OK:     {(df_stats['status']=='ok').sum()}")
print(f"   Células cache:  {(df_stats['status']=='cached').sum()}")
print(f"   Células falhas: {(df_stats['status']=='failed').sum()}")
print(f"   Total pixels:   {df_stats['n_pixels'].sum():,}")
print(f"   Média por célula: {df_stats['n_pixels'].mean():,.0f}")

In [ ]:
# ============================================================================
# ANÁLISE DE TRANSIÇÕES — frequência por célula e bioma (v2)
# ============================================================================
# Processos identificados por pixel (11 flags decompostos):
#
# Grupo 1 — Saída de N:
#   N_para_P:   N→Pastagem  (expansão de pastagem)
#   N_para_A:   N→Agricultura (conversão direta)
#   N_para_out: N→U/W/T (outras)
#
# Grupo 2 — Intensificação:
#   P_para_A:   Pastagem→Agricultura
#
# Grupo 3 — Estabilidade:
#   estavel:    >= 21 anos na mesma classe (> metade da série de 40 anos)
#
# Grupo 4 — Alternância N↔X (decomposta por destino):
#   alt_N_W:    N↔Água   → pulso hídrico       (Pantanal)
#   alt_N_A:    N↔Agric  → pousio agrícola      (Pampa)
#   alt_N_P:    N↔Past   → conversão incompleta (Cerrado)
#
# Grupo 5 — Retorno a N (censurado à direita, decomposto):
#   W_para_N:   Água→N   → retração hídrica     (Pantanal)
#   A_para_N:   Agric→N  → pousio retorna       (Pampa)
#   P_para_N:   Past→N   → regeneração           (Cerrado)
# ============================================================================

THRESHOLD_ESTAB = 21   # anos — mais da metade da série de 40 anos

# Códigos das classes agregadas: N=1, P=2, A=3, U=4, W=5, T=6
_N, _P, _A, _U, _W, _T = 1, 2, 3, 4, 5, 6

PROCESSOS = [
    'N_para_P', 'N_para_A', 'N_para_out',   # grupo 1
    'P_para_A',                               # grupo 2
    'estavel',                                # grupo 3
    'alt_N_W', 'alt_N_A', 'alt_N_P',         # grupo 4
    'W_para_N', 'A_para_N', 'P_para_N'       # grupo 5
]

PROC_LABELS = {
    'N_para_P':   'N→P (exp. pastagem)',
    'N_para_A':   'N→A (conv. direta)',
    'N_para_out': 'N→U/W (outras)',
    'P_para_A':   'P→A (intensif.)',
    'estavel':    'Estável (>=21 anos)',
    'alt_N_W':    'N↔W (pulso hídrico)',
    'alt_N_A':    'N↔A (pousio)',
    'alt_N_P':    'N↔P (conv. incompleta)',
    'W_para_N':   'W→N (retração hídr.)',
    'A_para_N':   'A→N (pousio retorna)',
    'P_para_N':   'P→N (regeneração)',
}


def classificar_pixel(serie):
    """
    Classifica processos ecológicos de um pixel.
    serie: array de classes por ano (uint8, 0=nodata)

    Alternância decomposta por destino de N:
      alt_N_W → pulso hídrico (Pantanal)
      alt_N_A → pousio agrícola (Pampa)
      alt_N_P → conversão incompleta/rebrota (Cerrado)

    Retorno a N decomposto por origem:
      W_para_N → retração hídrica (Pantanal)
      A_para_N → pousio retorna (Pampa)
      P_para_N → regeneração após pastagem
    """
    valid = serie[serie > 0]
    if len(valid) < 5:
        return None

    result = {p: False for p in PROCESSOS}
    result['classe_inicial'] = int(valid[0])
    result['classe_final']   = int(valid[-1])
    result['n_anos_validos'] = len(valid)

    # ── Grupo 1: primeira saída de N ──────────────────────────────────────
    if valid[0] == _N:
        for v in valid[1:]:
            if v == _T: continue
            if v == _P:   result['N_para_P']   = True; break
            elif v == _A: result['N_para_A']   = True; break
            elif v != _N: result['N_para_out'] = True; break

    # ── Grupo 2: P → A ────────────────────────────────────────────────────
    in_P = False
    for v in valid:
        if v == _P:               in_P = True
        elif v == _A and in_P:    result['P_para_A'] = True; break
        elif v not in (_P, _T):   in_P = False

    # ── Grupo 3: estabilidade ─────────────────────────────────────────────
    for cls in [_N, _P, _A, _U, _W]:
        if (valid == cls).sum() >= THRESHOLD_ESTAB:
            result['estavel'] = True
            break

    # ── Grupo 4: alternância N↔X decomposta ──────────────────────────────
    # Contar transições de saída de N por tipo de destino
    alt_counts = {_W: 0, _A: 0, _P: 0}
    prev = valid[0]
    for v in valid[1:]:
        if v == _T: continue
        if prev == _N and v in alt_counts:    alt_counts[v] += 1
        elif v == _N and prev in alt_counts:  alt_counts[prev] += 1
        prev = v
    # >= 3 transições = pelo menos 1.5 ciclos completos
    if alt_counts[_W] >= 3: result['alt_N_W'] = True
    if alt_counts[_A] >= 3: result['alt_N_A'] = True
    if alt_counts[_P] >= 3: result['alt_N_P'] = True

    # ── Grupo 5: retorno a N — decomposto por classe de origem ───────────
    if valid[-1] == _N and valid[0] != _N:
        # Última classe antes da sequência final de N
        ultima = None
        for v in reversed(valid[:-1]):
            if v != _N and v != _T:
                ultima = v; break
        if ultima == _W:   result['W_para_N'] = True
        elif ultima == _A: result['A_para_N'] = True
        elif ultima == _P: result['P_para_N'] = True

    return result


# ── Analisar células ──────────────────────────────────────────────────────────
SERIES_DIR = OUT_DIR / "series_temporais"
print(f"Analisando processos de transição (v2)...")
print(f"  Threshold estabilidade: {THRESHOLD_ESTAB} anos")
print(f"  Processos: {len(PROCESSOS)}")
all_processes = []

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample),
                   desc="Análise"):
    cell_id  = row['CELL_ID']
    npz_path = SERIES_DIR / f"{cell_id}.npz"
    if not npz_path.exists(): continue

    data   = np.load(npz_path)
    series = data['series']   # (n_pixels, n_anos)

    n_sample = min(1000, len(series))
    idx = np.random.choice(len(series), n_sample, replace=False)

    cell_proc = {'cell_id': cell_id, 'bioma': row['bioma'],
                 'bioma_dom': row['bioma'].split('-')[0],
                 'n_analisados': n_sample}
    for proc in PROCESSOS:
        cell_proc[proc] = 0

    for i in idx:
        r = classificar_pixel(series[i])
        if r is None: continue
        for proc in PROCESSOS:
            if r[proc]: cell_proc[proc] += 1

    all_processes.append(cell_proc)

df_proc = pd.DataFrame(all_processes)

# Normalizar
for proc in PROCESSOS:
    df_proc[f'{proc}_pct'] = 100 * df_proc[proc] / df_proc['n_analisados']

df_proc.to_csv(OUT_DIR / "process_analysis_v2.csv", index=False)

print(f"\n✅ Análise v2 concluída!")
print(f"\nFrequência média dos processos por célula:")
print(f"{'Processo':<25} {'% médio':>9} {'% min':>7} {'% max':>7}")
print("-" * 52)
for proc in PROCESSOS:
    col = f'{proc}_pct'
    label = PROC_LABELS[proc]
    print(f"{label:<25} {df_proc[col].mean():>8.1f}% "
          f"{df_proc[col].min():>6.1f}% {df_proc[col].max():>6.1f}%")


In [ ]:
# ============================================================================
# VISUALIZAÇÃO — distribuição espacial das células e processos
# ============================================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

cores_bioma = {'A':'#27ae60','C':'#f39c12','M':'#2980b9',
               'CA':'#e74c3c','PP':'#8e44ad','P':'#16a085'}

# Mapa das células sorteadas
for _, row in df_sample.iterrows():
    dom = row['bioma'].split('-')[0]
    cor = cores_bioma.get(dom, 'gray')
    cx  = (row['LON_MIN'] + row['LON_MAX']) / 2
    cy  = (row['LAT_MIN'] + row['LAT_MAX']) / 2
    axes[0,0].scatter(cx, cy, c=cor, s=60, alpha=0.8)
    axes[0,0].add_patch(plt.Rectangle(
        (row['LON_MIN'], row['LAT_MIN']), 2, 2,
        fill=False, edgecolor=cor, linewidth=0.8, alpha=0.5))

for bioma, cor in cores_bioma.items():
    axes[0,0].scatter([], [], c=cor, s=60, label=bioma)
axes[0,0].set_xlabel('Longitude')
axes[0,0].set_ylabel('Latitude')
axes[0,0].set_title(f'Células Sorteadas (n={len(df_sample)})',
                     fontweight='bold')
axes[0,0].legend(title='Bioma', fontsize=8)
axes[0,0].set_xlim(-75, -33); axes[0,0].set_ylim(-35, 7)

# Frequência dos processos por bioma
proc_cols = ['N_para_P_pct','N_para_A_pct','P_para_A_pct',
             'alt_N_W_pct','alt_N_A_pct','alt_N_P_pct',
             'W_para_N_pct','A_para_N_pct','P_para_N_pct']
proc_short = ['N→P','N→A','P→A','N↔W','N↔A','N↔P','W→N','A→N','P→N']

if len(df_proc) > 0:
    for ax, proc, label in zip(
        [axes[0,1],axes[0,2],axes[1,0],axes[1,1],
         axes[1,2],axes[0,1],axes[0,2],axes[1,0],axes[1,1]],
        proc_cols, proc_short
    ):
        if proc not in df_proc.columns: continue
        df_by_bioma = df_proc.groupby('bioma_dom')[proc].mean()
        bars = ax.bar(df_by_bioma.index,
                      df_by_bioma.values,
                      color=[cores_bioma.get(b,'gray')
                             for b in df_by_bioma.index],
                      alpha=0.8, edgecolor='white')
        ax.set_ylabel('% pixels')
        ax.set_title(f'Processo: {label}', fontweight='bold')
        for bar in bars:
            ax.text(bar.get_x()+bar.get_width()/2,
                    bar.get_height()+0.5,
                    f'{bar.get_height():.1f}%',
                    ha='center', fontsize=9)

plt.suptitle('GeoFM — Análise de Processos de Transição por Bioma',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUT_DIR / 'geofm_processes_by_biome.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ geofm_processes_by_biome.png")

# Salvar metadata
meta = {
    'version':       'GeoFM_v0.1_sampling',
    'date':          datetime.now().strftime("%Y-%m-%d %H:%M"),
    'n_cells':       len(df_sample),
    'seed':          SEED,
    'min_coverage':  MIN_COV,
    'years':         [int(y) for y in YEARS],
    'n_years':       len(YEARS),
    'class_map':     {str(k):v for k,v in CLASS_MAP.items()},
    'class_codes':   CLASS_CODES,
    'processes': [
        'N_to_P (pasture expansion)',
        'N_to_A (direct agriculture conversion)',
        'N_to_other (other non-native)',
        'P_to_A (agricultural intensification)',
        'stability (>=21 years same class)',
        'alt_N_W (N-Water alternation — hydrological pulse, Pantanal)',
        'alt_N_A (N-Agriculture alternation — fallow, Pampa)',
        'alt_N_P (N-Pasture alternation — incomplete conversion, Cerrado)',
        'W_to_N (water receding — Pantanal)',
        'A_to_N (fallow returning to native — Pampa)',
        'P_to_N (regeneration after pasture — Cerrado)'
    ],
    'threshold_estab_anos': 21,
    'note_class21': (
        'Class 21 (mosaic) kept as T (transition) — same approach as P->S. '
        'Not aggregated into separate M class. '
        'Class 25 aggregated into U (non-vegetated).'
    ),
    'raster_source': 'MapBiomas Collection 10.1',
    'raster_pattern': RASTER_PAT,
    'foundation_model_note': (
        'This dataset is designed as pre-training data for a '
        'geospatial foundation model based on LULC time series. '
        'Stratified sampling ensures representation of all Brazilian biomes. '
        'The 5 processes of interest serve as downstream fine-tuning tasks.'
    )
}
with open(OUT_DIR / 'geofm_dataset_metadata.json', 'w',
          encoding='utf-8') as f:
    json.dump(meta, f, indent=2)
print("✅ geofm_dataset_metadata.json")

print(f"\n{'='*60}")
print(f"GeoFM DATASET — FASE 1 COMPLETA")
print(f"{'='*60}")
print(f"  Células: {len(df_sample):,}")
print(f"  Output: {OUT_DIR}")
print(f"  Séries: {SERIES_DIR}")
print(f"\n🎯 Próximo passo: definir objetivo de pré-treino")
print(f"   (masked prediction / process prediction)")